# Copositive QA-abstention on a real LLM (Qwen3-0.6B)

Certified (absolute-threshold) vs calibrated (softmax-confidence) abstention under distribution shift.
**GPU preset. Run all.** Part 1 ≈10 min; Part 2 (heavier LoRA, resumable) ≈1.5–2 h.


In [ ]:
# 1. deps
!pip -q install -U transformers datasets

In [ ]:
!wget -q https://raw.githubusercontent.com/areidyOTH/copositive-colab/main/qa_abstention.py -O qa_abstention.py
print('got qa_abstention.py')

## Part 1 — frozen backbone (R14): only a read head trained on fixed embeddings


In [ ]:
MODEL = 'Qwen/Qwen3-0.6B'   # fallback if it errors: 'Qwen/Qwen2.5-0.5B'
for s in range(3):
    print(f'\n========== seed {s} ==========')
    !python qa_abstention.py --model {MODEL} --device cuda --pooling last --n_train 4000 --n_eval 2000 --steps 600 --seed {s}

## Part 2 — heavier LoRA-adapt (resumable)

Lets the **embeddings** adapt: 1500 LoRA steps, rank 32, 2 seeds, both reads (~1.5–2 h on T4). It
**checkpoints every ~20 min** to `artifacts/` and **resumes** if re-run (completed stages skipped, an
interrupted LoRA stage continues from its last step).

**To survive a closed tab / disconnect: run this notebook with `Save Version → Save & Run All
(Commit)`** — it runs headless to completion and persists output. If it ever dies early, just run the
notebook again on the saved output and it picks up where it left off.


In [ ]:
# deps for LoRA. Kaggle ships an old torchao that makes peft error on LoRA -> remove it (unused).
!pip -q install -U peft
!pip uninstall -y torchao

In [ ]:
!wget -q https://raw.githubusercontent.com/areidyOTH/copositive-colab/main/qa_abstention.py -O qa_abstention.py
!wget -q https://raw.githubusercontent.com/areidyOTH/copositive-colab/main/qa_abstention_lora.py -O qa_abstention_lora.py
print('got qa_abstention_lora.py')

In [ ]:
# heavier LoRA, resumable. Re-running this cell resumes from artifacts/ if interrupted.
!python qa_abstention_lora.py --model {MODEL} --device cuda --pooling last

In [ ]:
# zip the trained models. Colab auto-downloads; on Kaggle grab artifacts.zip from the Output panel.
!zip -qr artifacts.zip artifacts
try:
    from google.colab import files; files.download('artifacts.zip')
except Exception:
    from IPython.display import FileLink; display(FileLink('artifacts.zip'))
    print('artifacts.zip ready in', __import__('os').getcwd())